# ISIC 2016 — StyleGAN2-ADA + DASS: chạy pipeline trên Colab

Notebook này **không chứa logic** — mọi code nằm trong package `isic_dass` (thư mục `src/`).
Mỗi cell gọi một stage qua CLI `isic-dass`, mỗi stage chạy trong **tiến trình riêng** nên PyTorch (GAN)
và TensorFlow (classifier) không tranh VRAM. Mọi stage đều **chạy lại được**: artefact lưu trên Drive,
phần đã xong sẽ được bỏ qua / khôi phục.

Thứ tự: `prepare → gan-setup → gan-train → gan-report → generate → (frequency) → select → train → aggregate → quality`

Runtime: **GPU** (A100/L4 khuyến nghị; T4 chạy được nhưng GAN chậm).

## 0. Kết nối Drive, lấy mã nguồn và cài đặt

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os

# Cách 1 (khuyến nghị): clone từ GitHub — điền URL repo sau khi đã push
REPO_URL = ""   # ví dụ "https://github.com/<user>/isic-dass.git"
# Cách 2: thư mục dự án đã upload lên Google Drive
PROJECT_ON_DRIVE = "/content/drive/MyDrive/isic-dass"

PROJECT_DIR = "/content/isic-dass"
if REPO_URL:
    if os.path.isdir(PROJECT_DIR):
        !git -C {PROJECT_DIR} pull -q
    else:
        !git clone -q {REPO_URL} {PROJECT_DIR}
else:
    !rsync -a --delete --exclude .git "{PROJECT_ON_DRIVE}/" {PROJECT_DIR}/

%cd {PROJECT_DIR}
# torch và tensorflow đã có sẵn trên Colab -> chỉ cài package + phụ thuộc nhẹ
!pip install -q -e . ninja click keras-hub

In [ ]:
CONFIG = "configs/default.yaml"      # đổi sang configs/smoke.yaml để chạy thử nhanh
!isic-dass --config {CONFIG} show-config

In [ ]:
# Tiện ích hiển thị hình / bảng mà các stage đã lưu lên Drive
import glob, json
import pandas as pd
from IPython.display import Image, display

_out = !isic-dass --config {CONFIG} show-config
cfg = json.loads("\n".join(_out))
DRIVE = cfg["paths"]["drive_root"]
RESULTS = f"{DRIVE}/results_{cfg['paths']['run_tag']}"

def show(pattern):
    for p in sorted(glob.glob(f"{RESULTS}/{pattern}")):
        print(p)
        display(Image(p))

def table(name):
    display(pd.read_csv(f"{RESULTS}/{name}"))

## 1. Dữ liệu: copy từ Drive, tiền xử lý PNG 256×256, tách val cố định

In [ ]:
!isic-dass --config {CONFIG} prepare
show("class_distribution.png")

## 2. StyleGAN2-ADA có điều kiện

`gan-setup` clone repo NVlabs và vá cho PyTorch 2.x. `gan-train` train với early stopping theo KID của lớp
thiểu số; **Colab ngắt thì chạy lại đúng cell này** để resume từ `latest.pkl`.
Chỉ dùng `--fresh-start` khi muốn XOÁ GAN cũ (ví dụ sau khi đổi `gan.gamma` / `gan.cfg`).

In [ ]:
!isic-dass --config {CONFIG} gan-setup

In [ ]:
!isic-dass --config {CONFIG} gan-train

In [ ]:
!isic-dass --config {CONFIG} gan-report
show("gan/*.png")

## 3. Candidate pool (sinh hoặc khôi phục từ Drive)

In [ ]:
!isic-dass --config {CONFIG} generate

## 4. (Tuỳ chọn) Fingerprint miền tần số

Bật chuẩn hoá / mitigation bằng `--set frequency.match_resize_chain=true` hoặc
`--set frequency.apply_spectral_mitigation=true` (phải dùng cùng cờ ở các stage sau).

In [ ]:
!isic-dass --config {CONFIG} frequency
table("frequency_fingerprint_auc.csv")
show("frequency_analysis/*.png")

## 5. DASS: nhúng E_v / E_d, chấm điểm, chọn ảnh cho M0–M6

In [ ]:
!isic-dass --config {CONFIG} select
table("probe_auc_Ev_vs_Ed.csv")
table("shortcut_check.csv")
show("selection_figures/scatter_Mv_Md.png")
show("selection_figures/grid_M6_dass_*.png")

## 6. Train classifier

Mỗi cell độc lập, dừng / chạy lại được — lần chạy đã có file dự đoán trên Drive sẽ bị bỏ qua.
Model: `EfficientNetV2B0`, `ResNet50`, `DenseNet121`, `ConvNeXtTiny`, `ViT-B16`, `SwinT`.

In [ ]:
!isic-dass --config {CONFIG} train --model EfficientNetV2B0 --seeds 2026 2027 2028

In [ ]:
!isic-dass --config {CONFIG} train --model ResNet50 --seeds 2026 2027 2028

In [ ]:
!isic-dass --config {CONFIG} train --model DenseNet121 --seeds 2026 2027 2028

In [ ]:
!isic-dass --config {CONFIG} train --model ConvNeXtTiny --seeds 2026 2027 2028

In [ ]:
!isic-dass --config {CONFIG} train --model SwinT --seeds 2026 2027 2028

## 7. Gộp kết quả và chất lượng ảnh sinh

In [ ]:
!isic-dass --config {CONFIG} aggregate
table("summary_mean_std.csv")

In [ ]:
!isic-dass --config {CONFIG} quality